In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models
datapath = "../../../desktop/quant/hist/aaplIntra.csv"

In [2]:
df = pd.read_csv(datapath)

In [3]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [ ]:
# 1-step ahead log-return target from Close
df["log_close"] = np.log(df["Close"])
df["y"] = df["log_close"].shift(-1) - df["log_close"]

# Simple lag features
for k in [1, 2, 3, 5]:
    df[f"ret_lag_{k}"] = df["log_close"].diff(k)

# Drop last row and any NAs from lags
df = df.dropna()

feature_cols = [c for c in df.columns if c.startswith("ret_lag_")]
X = df[feature_cols].values.astype("float32")
y = df["y"].values.astype("float32")

In [5]:
# train validation test split

split = int(0.8 * len(df))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

In [15]:
n_features = X_train.shape[1]

model = models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(
        1,
        activation=None,  # linear
        kernel_regularizer=regularizers.l2(1e-4)  # ridge penalty
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    verbose=1
)

Epoch 1/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 2s 463us/step - loss: 4.9676e-05 - val_loss: 1.1511e-07
Epoch 2/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 2s 432us/step - loss: 1.0240e-07 - val_loss: 7.8721e-08
Epoch 3/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 413us/step - loss: 1.1888e-07 - val_loss: 7.9464e-08
Epoch 4/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 412us/step - loss: 9.8858e-08 - val_loss: 8.8339e-08
Epoch 5/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 414us/step - loss: 1.0188e-07 - val_loss: 7.7168e-08
Epoch 6/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 414us/step - loss: 1.0344e-07 - val_loss: 7.8654e-08
Epoch 7/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 411us/step - loss: 1.0442e-07 - val_loss: 8.1008e-08
Epoch 8/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 410us/step - loss: 1.0262e-07 - val_loss: 7.4930e-08
Epoch 9/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 413us/step - loss: 9.4583e-08 - val_loss: 8.0457e-08
Epoch 10/50
3514/3514 ━━━━━━━━━━━━━━━━━━━━ 1s 409us/step - loss: 1.0678e-07 - val_loss: 7.4595e-08
Epoch 11/50
3514/35

In [16]:
model.save('models/regularizedLinear.keras')